# Adaptive RAG Implementation

## Introduction

In this notebook, I am implementing a simple **Adaptive RAG (Retrieval-Augmented Generation)** system.

The main idea of Adaptive RAG is that the system does not use the same retrieval method for every question. First, it looks at the question and selects a suitable strategy.

For example, a simple question can use normal retrieval, while a vague question can be rewritten before retrieval. A question that needs different perspectives can use multiple queries, and a complex question can be divided into smaller sub-questions.

After retrieval, the system also uses reranking and checks the quality of the retrieved information before generating the final answer.

### Main steps in this notebook

1. Create a small knowledge base
2. Split documents into smaller chunks
3. Generate embeddings using Sentence Transformers
4. Create a FAISS vector index
5. Create a BM25 keyword index
6. Perform hybrid search using FAISS and BM25
7. Combine results using Reciprocal Rank Fusion (RRF)
8. Use a Gemini-based adaptive router
9. Select a retrieval strategy
10. Rewrite queries when required
11. Generate multiple queries when required
12. Decompose complex questions when required
13. Rerank the retrieved documents
14. Evaluate the retrieval quality
15. Generate the final answer
16. Evaluate the final answer

### Adaptive RAG Strategies

- **DIRECT** – used for simple conversational questions
- **SIMPLE** – normal hybrid retrieval
- **REWRITE** – rewrites unclear questions before retrieval
- **MULTI_QUERY** – creates multiple search queries
- **DECOMPOSE** – breaks a complex question into smaller questions

### Architecture

```text
                         User Question
                               |
                               v
                       Adaptive Router
                               |
          +----------+---------+---------+----------+
          |          |         |         |          |
          v          v         v         v          v
       DIRECT     SIMPLE    REWRITE   MULTI-QUERY  DECOMPOSE
          |          |         |         |          |
          |          +---------+---------+----------+
          |                    |
          |                    v
          |             Hybrid Retrieval
          |              /             \
          |           FAISS            BM25
          |              \             /
          |               \           /
          |                  RRF
          |                   |
          |                   v
          |               Reranking
          |                   |
          |                   v
          |          Retrieval Evaluation
          |                   |
          +-------------------+
                      |
                      v
                 Answer Generation
                      |
                      v
                Answer Evaluation
                      |
                      v
                  Final Answer

In [76]:
!pip -q install -U sentence-transformers faiss-cpu google-genai rank-bm25

In [77]:
import os
import re
import json
import numpy as np
import faiss

from getpass import getpass
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from google import genai

print("Libraries imported successfully.")

Libraries imported successfully.


In [78]:
GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

MODEL_NAME = "gemini-3.5-flash-lite"

print("Gemini initialized successfully.")
print("Model:", MODEL_NAME)

Enter your Gemini API key: ··········
Gemini initialized successfully.
Model: gemini-3.5-flash-lite


In [79]:
documents = [

    {
        "id": "doc1",
        "title": "RAG Introduction",
        "text": """
        Retrieval-Augmented Generation, commonly called RAG, combines
        information retrieval with large language models. RAG retrieves
        relevant information from an external knowledge base and provides
        that information to the language model as context.
        """
    },

    {
        "id": "doc2",
        "title": "Vector Databases",
        "text": """
        A vector database stores numerical representations of data called
        embeddings. These embeddings are used for semantic similarity search.
        A question can be converted into an embedding and compared with
        document embeddings.
        """
    },

    {
        "id": "doc3",
        "title": "Embeddings",
        "text": """
        Embeddings are numerical vectors that represent the semantic meaning
        of text. Similar pieces of text tend to have embeddings that are close
        to each other in vector space. Embeddings are commonly used for
        semantic search and RAG.
        """
    },

    {
        "id": "doc4",
        "title": "Semantic Search",
        "text": """
        Semantic search retrieves information based on meaning instead of
        only exact keyword matching. The query and documents are converted
        into embeddings and their similarity is calculated.
        """
    },

    {
        "id": "doc5",
        "title": "BM25 Keyword Search",
        "text": """
        BM25 is a keyword-based information retrieval algorithm. It is useful
        for exact keywords, technical terminology, identifiers and error codes.
        """
    },

    {
        "id": "doc6",
        "title": "Hybrid Search",
        "text": """
        Hybrid search combines semantic vector search and lexical keyword
        search. Vector search is useful for meaning while keyword search is
        useful for exact terms.
        """
    },

    {
        "id": "doc7",
        "title": "Reranking",
        "text": """
        Reranking is a second-stage retrieval process. An initial retriever
        finds candidate documents and a reranker gives them new relevance
        scores to improve the final ranking.
        """
    },

    {
        "id": "doc8",
        "title": "Authentication Errors",
        "text": """
        Authentication errors can happen when credentials are invalid or
        authentication tokens expire.

        ERR-401 indicates an authentication failure.

        ERR-403 indicates that the user is authenticated but does not have
        permission to access a resource.

        Access tokens authenticate requests and refresh tokens can be used
        to obtain new access tokens.
        """
    },

    {
        "id": "doc9",
        "title": "Query Rewriting",
        "text": """
        Query rewriting changes the original question into a clearer and more
        retrieval-friendly query. It can make vague concepts explicit and
        add useful terminology without changing the original meaning.
        """
    },

    {
        "id": "doc10",
        "title": "Query Expansion",
        "text": """
        Query expansion creates alternative versions of a search query.
        Different words or related terminology can improve the chance of
        finding relevant information.
        """
    },

    {
        "id": "doc11",
        "title": "Multi-Query Retrieval",
        "text": """
        Multi-query retrieval uses several search queries for the same
        information need. The results from different queries are combined
        to improve retrieval coverage.
        """
    },

    {
        "id": "doc12",
        "title": "Query Decomposition",
        "text": """
        Query decomposition breaks a complex question into smaller
        independent questions. Each smaller question can be searched
        separately and the evidence can then be combined.
        """
    },

    {
        "id": "doc13",
        "title": "Contextual Retrieval",
        "text": """
        Contextual retrieval adds information about where a document chunk
        came from and how it relates to the larger document.
        """
    },

    {
        "id": "doc14",
        "title": "Self-RAG",
        "text": """
        Self-RAG allows a system to evaluate whether retrieval is needed,
        whether retrieved evidence is relevant, and whether the generated
        answer is supported by the evidence.
        """
    },

    {
        "id": "doc15",
        "title": "Corrective RAG",
        "text": """
        Corrective RAG evaluates retrieved documents before using them.
        If retrieval is poor, the system can rewrite the query and retrieve
        again or use another retrieval strategy.
        """
    },

    {
        "id": "doc16",
        "title": "Adaptive RAG",
        "text": """
        Adaptive RAG dynamically selects a retrieval strategy according to
        the type and complexity of the user's question. It can use direct
        answering, normal retrieval, query rewriting, multiple queries,
        or query decomposition.
        """
    }
]

print("Number of documents:", len(documents))

Number of documents: 16


In [80]:
def chunk_text(text, chunk_size=70, overlap=15):

    words = text.split()
    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(
            words[start:end]
        )

        if chunk.strip():
            chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks

In [81]:
chunks = []

for document in documents:

    document_chunks = chunk_text(
        document["text"]
    )

    for number, chunk in enumerate(
        document_chunks
    ):

        chunks.append({
            "chunk_id": f'{document["id"]}_chunk_{number}',
            "document_id": document["id"],
            "title": document["title"],
            "chunk_number": number,
            "text": chunk
        })

print("Total chunks:", len(chunks))

Total chunks: 16


In [82]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_dimension = (
    embedding_model.get_sentence_embedding_dimension()
)

print("Embedding dimension:", embedding_dimension)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384


/tmp/ipykernel_5093/2176670758.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [83]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

vector_index = faiss.IndexFlatIP(
    embedding_dimension
)

vector_index.add(embeddings)

print("FAISS vectors:", vector_index.ntotal)

FAISS vectors: 16


In [84]:
def tokenize(text):

    return re.findall(
        r"\b\w+\b",
        text.lower()
    )


tokenized_chunks = [
    tokenize(chunk["text"])
    for chunk in chunks
]

bm25 = BM25Okapi(
    tokenized_chunks
)

print("BM25 index created.")

BM25 index created.


In [85]:
def vector_search(query, top_k=10):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = vector_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, index) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        result = chunks[index].copy()

        result["vector_score"] = float(score)
        result["vector_rank"] = rank

        results.append(result)

    return results

In [86]:
def bm25_search(query, top_k=10):

    scores = bm25.get_scores(
        tokenize(query)
    )

    indices = np.argsort(
        scores
    )[::-1][:top_k]

    results = []

    for rank, index in enumerate(
        indices,
        start=1
    ):

        result = chunks[index].copy()

        result["bm25_score"] = float(
            scores[index]
        )

        result["bm25_rank"] = rank

        results.append(result)

    return results

In [87]:
def reciprocal_rank_fusion(
    result_lists,
    k=60
):

    fused = {}

    for results in result_lists:

        for rank, result in enumerate(
            results,
            start=1
        ):

            chunk_id = result["chunk_id"]

            if chunk_id not in fused:

                fused[chunk_id] = {
                    "chunk": result,
                    "score": 0.0
                }

            fused[chunk_id]["score"] += (
                1 / (k + rank)
            )

    ranked = sorted(
        fused.values(),
        key=lambda x: x["score"],
        reverse=True
    )

    final_results = []

    for item in ranked:

        result = item["chunk"].copy()

        result["rrf_score"] = item["score"]

        final_results.append(result)

    return final_results

In [88]:
def hybrid_search(query, top_k=8):

    vector_results = vector_search(
        query,
        top_k=10
    )

    bm25_results = bm25_search(
        query,
        top_k=10
    )

    fused_results = reciprocal_rank_fusion(
        [
            vector_results,
            bm25_results
        ]
    )

    return fused_results[:top_k]

In [89]:
question = "What does ERR-401 mean?"

results = hybrid_search(
    question,
    top_k=5
)

for i, result in enumerate(
    results,
    start=1
):

    print("-" * 60)
    print("Rank:", i)
    print("Title:", result["title"])
    print("RRF Score:", round(
        result["rrf_score"],
        4
    ))
    print(result["text"])

------------------------------------------------------------
Rank: 1
Title: Authentication Errors
RRF Score: 0.0328
Authentication errors can happen when credentials are invalid or authentication tokens expire. ERR-401 indicates an authentication failure. ERR-403 indicates that the user is authenticated but does not have permission to access a resource. Access tokens authenticate requests and refresh tokens can be used to obtain new access tokens.
------------------------------------------------------------
Rank: 2
Title: Corrective RAG
RRF Score: 0.0315
Corrective RAG evaluates retrieved documents before using them. If retrieval is poor, the system can rewrite the query and retrieve again or use another retrieval strategy.
------------------------------------------------------------
Rank: 3
Title: Self-RAG
RRF Score: 0.0315
Self-RAG allows a system to evaluate whether retrieval is needed, whether retrieved evidence is relevant, and whether the generated answer is supported by the evid

In [90]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded.


In [91]:
def rerank_documents(
    query,
    results,
    top_k=5
):

    if not results:
        return []

    pairs = [
        [query, result["text"]]
        for result in results
    ]

    scores = reranker.predict(
        pairs
    )

    reranked = []

    for result, score in zip(
        results,
        scores
    ):

        item = result.copy()

        item["reranker_score"] = float(
            score
        )

        reranked.append(item)

    reranked.sort(
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    return reranked[:top_k]

In [92]:
def adaptive_router(question):

    prompt = f"""
You are an Adaptive RAG strategy router.

Choose exactly one strategy.

DIRECT:
Use for simple conversational questions.

SIMPLE:
Use for normal factual questions.

REWRITE:
Use when the question is vague or unclear.

MULTI_QUERY:
Use when different perspectives may help retrieval.

DECOMPOSE:
Use for complex questions with multiple parts.

Return ONLY JSON:

{{
    "strategy": "DIRECT",
    "reason": "short explanation"
}}

QUESTION:
{question}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    try:

        result = json.loads(text)

        strategy = result.get(
            "strategy",
            "SIMPLE"
        ).upper()

        valid_strategies = [
            "DIRECT",
            "SIMPLE",
            "REWRITE",
            "MULTI_QUERY",
            "DECOMPOSE"
        ]

        if strategy not in valid_strategies:
            strategy = "SIMPLE"

        return {
            "strategy": strategy,
            "reason": result.get(
                "reason",
                ""
            )
        }

    except Exception:

        return {
            "strategy": "SIMPLE",
            "reason": "Router could not be parsed."
        }

In [93]:
def rewrite_query(question):

    prompt = f"""
Rewrite this question into a clearer
retrieval-friendly query.

Rules:
1. Keep the original meaning.
2. Make vague concepts clear.
3. Add useful terminology if needed.
4. Do not invent facts.
5. Keep it short.

Return ONLY JSON:

{{
    "rewritten_query": "...",
    "reason": "short explanation"
}}

QUESTION:
{question}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    try:

        result = json.loads(text)

        return {
            "rewritten_query": result.get(
                "rewritten_query",
                question
            ),
            "reason": result.get(
                "reason",
                ""
            )
        }

    except Exception:

        return {
            "rewritten_query": question,
            "reason": "Original question retained."
        }

In [94]:
def generate_multi_queries(
    question,
    num_queries=2
):

    prompt = f"""
Generate {num_queries} different search queries
for the same information need.

Use different useful perspectives.

Do not answer the question.

Return ONLY JSON:

{{
    "queries": [
        "query 1",
        "query 2"
    ]
}}

QUESTION:
{question}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    try:

        result = json.loads(text)

        queries = result.get(
            "queries",
            []
        )

        queries = [
            q.strip()
            for q in queries
            if isinstance(q, str)
            and q.strip()
        ]

        return queries[:num_queries]

    except Exception:

        return [question]

In [95]:
def decompose_query(
    question,
    max_questions=3
):

    prompt = f"""
Break this question into smaller
independent questions.

Each question should cover one important part.

If the question is simple, return one question.

Return ONLY JSON:

{{
    "sub_questions": [
        "question 1",
        "question 2",
        "question 3"
    ]
}}

QUESTION:
{question}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    try:

        result = json.loads(text)

        questions = result.get(
            "sub_questions",
            []
        )

        questions = [
            q.strip()
            for q in questions
            if isinstance(q, str)
            and q.strip()
        ]

        return questions[:max_questions]

    except Exception:

        return [question]

In [96]:
def multi_query_retrieval(
    queries,
    top_k=5
):

    result_lists = []

    for query in queries:

        result_lists.append(
            hybrid_search(
                query,
                top_k=5
            )
        )

    fused = reciprocal_rank_fusion(
        result_lists
    )

    return fused[:top_k]

In [97]:
def decomposed_retrieval(
    sub_questions,
    top_k=5
):

    result_lists = []

    for question in sub_questions:

        result_lists.append(
            hybrid_search(
                question,
                top_k=5
            )
        )

    fused = reciprocal_rank_fusion(
        result_lists
    )

    return fused[:top_k]

In [98]:
def build_context(results):

    context = ""

    for i, result in enumerate(
        results,
        start=1
    ):

        context += f"""
SOURCE {i}
TITLE: {result["title"]}

{result["text"]}

"""

    return context

In [99]:
def evaluate_retrieval(
    question,
    results
):

    context = build_context(results)

    prompt = f"""
Evaluate the quality of the retrieved evidence.

Return ONLY JSON:

{{
    "grade": "GOOD",
    "score": 1.0,
    "reason": "short explanation"
}}

GOOD = directly useful
AMBIGUOUS = partly useful
BAD = mostly irrelevant

QUESTION:
{question}

CONTEXT:
{context}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    try:

        result = json.loads(text)

        grade = result.get(
            "grade",
            "AMBIGUOUS"
        ).upper()

        if grade not in [
            "GOOD",
            "AMBIGUOUS",
            "BAD"
        ]:
            grade = "AMBIGUOUS"

        score = float(
            result.get(
                "score",
                0.0
            )
        )

        return {
            "grade": grade,
            "score": max(
                0.0,
                min(1.0, score)
            ),
            "reason": result.get(
                "reason",
                ""
            )
        }

    except Exception:

        return {
            "grade": "AMBIGUOUS",
            "score": 0.4,
            "reason": "Evaluation failed."
        }

In [100]:
def generate_answer(
    question,
    context
):

    prompt = f"""
You are an Adaptive RAG question-answering system.

Answer the question using the supplied evidence.

Rules:
1. Use the evidence as the main source.
2. Do not invent information.
3. If the evidence is insufficient, say so.
4. Give a clear and direct answer.

QUESTION:
{question}

EVIDENCE:
{context}

ANSWER:
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text.strip()

In [101]:
def evaluate_answer(
    question,
    answer,
    context
):

    prompt = f"""
Check whether the answer is supported
by the evidence.

Return ONLY JSON:

{{
    "supported": true,
    "score": 1.0,
    "reason": "short explanation"
}}

1.0 = fully supported
0.7 = mostly supported
0.4 = partially supported
0.0 = unsupported

QUESTION:
{question}

EVIDENCE:
{context}

ANSWER:
{answer}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    try:

        result = json.loads(text)

        score = float(
            result.get(
                "score",
                0.0
            )
        )

        score = max(
            0.0,
            min(1.0, score)
        )

        return {
            "supported": bool(
                result.get(
                    "supported",
                    score >= 0.7
                )
            ),
            "score": score,
            "reason": result.get(
                "reason",
                ""
            )
        }

    except Exception:

        return {
            "supported": False,
            "score": 0.0,
            "reason": "Answer evaluation failed."
        }

In [102]:
def adaptive_rag(
    question,
    candidate_k=8,
    final_k=4,
    max_corrections=1
):

    trace = []

    # Step 1: Select strategy
    routing = adaptive_router(
        question
    )

    strategy = routing["strategy"]

    trace.append({
        "step": "router",
        "strategy": strategy,
        "reason": routing["reason"]
    })

    # DIRECT strategy
    if strategy == "DIRECT":

        prompt = f"""
Answer this question directly and concisely:

{question}
"""

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        return {
            "question": question,
            "strategy": strategy,
            "answer": response.text.strip(),
            "sources": [],
            "trace": trace
        }

    # Step 2: Prepare retrieval query
    retrieval_query = question
    queries = [question]

    if strategy == "REWRITE":

        rewrite = rewrite_query(
            question
        )

        retrieval_query = rewrite[
            "rewritten_query"
        ]

        trace.append({
            "step": "rewrite",
            "query": retrieval_query
        })

    elif strategy == "MULTI_QUERY":

        queries = generate_multi_queries(
            question,
            num_queries=2
        )

        trace.append({
            "step": "multi_query",
            "queries": queries
        })

    elif strategy == "DECOMPOSE":

        queries = decompose_query(
            question,
            max_questions=3
        )

        trace.append({
            "step": "decomposition",
            "questions": queries
        })

    # Step 3: Retrieval
    if strategy in [
        "SIMPLE",
        "REWRITE"
    ]:

        candidates = hybrid_search(
            retrieval_query,
            top_k=candidate_k
        )

    elif strategy == "MULTI_QUERY":

        candidates = multi_query_retrieval(
            queries,
            top_k=candidate_k
        )

    else:

        candidates = decomposed_retrieval(
            queries,
            top_k=candidate_k
        )

    trace.append({
        "step": "retrieval",
        "count": len(candidates)
    })

    # Step 4: Reranking
    reranked = rerank_documents(
        retrieval_query,
        candidates,
        top_k=final_k
    )

    trace.append({
        "step": "reranking",
        "count": len(reranked)
    })

    # Step 5: Evaluate retrieval
    retrieval_eval = evaluate_retrieval(
        question,
        reranked
    )

    trace.append({
        "step": "retrieval_evaluation",
        "result": retrieval_eval
    })

    # Step 6: Correct if retrieval is poor
    correction_count = 0

    while (
        retrieval_eval["grade"] != "GOOD"
        and correction_count < max_corrections
    ):

        correction_count += 1

        correction = rewrite_query(
            question
        )

        retrieval_query = correction[
            "rewritten_query"
        ]

        trace.append({
            "step": "correction",
            "query": retrieval_query
        })

        candidates = hybrid_search(
            retrieval_query,
            top_k=candidate_k
        )

        reranked = rerank_documents(
            retrieval_query,
            candidates,
            top_k=final_k
        )

        retrieval_eval = evaluate_retrieval(
            question,
            reranked
        )

    # Step 7: Generate answer
    context = build_context(
        reranked
    )

    answer = generate_answer(
        question,
        context
    )

    trace.append({
        "step": "generation"
    })

    # Step 8: Evaluate answer
    answer_eval = evaluate_answer(
        question,
        answer,
        context
    )

    trace.append({
        "step": "answer_evaluation",
        "result": answer_eval
    })

    return {
        "question": question,
        "strategy": strategy,
        "routing_reason": routing["reason"],
        "answer": answer,
        "sources": reranked,
        "retrieval_grade": retrieval_eval["grade"],
        "retrieval_score": retrieval_eval["score"],
        "answer_supported": answer_eval["supported"],
        "answer_score": answer_eval["score"],
        "trace": trace
    }

In [103]:
router_questions = [
    "Hello, how are you?",
    "What is an embedding?",
    "What does that login thing mean?",
    "Why are semantic and keyword search both useful?"
]

for question in router_questions:

    result = adaptive_router(
        question
    )

    print("-" * 60)
    print("QUESTION:", question)
    print("STRATEGY:", result["strategy"])
    print("REASON:", result["reason"])

------------------------------------------------------------
QUESTION: Hello, how are you?
STRATEGY: DIRECT
REASON: The question is a simple conversational greeting.
------------------------------------------------------------
QUESTION: What is an embedding?
STRATEGY: SIMPLE
REASON: This is a straightforward factual question about a specific technical concept.
------------------------------------------------------------
QUESTION: What does that login thing mean?
STRATEGY: REWRITE
REASON: The question is vague and unclear because it lacks context about what 'that login thing' refers to.
------------------------------------------------------------
QUESTION: Why are semantic and keyword search both useful?
STRATEGY: SIMPLE
REASON: This is a normal factual question about information retrieval concepts.


In [104]:
question = "What does ERR-401 mean?"

result = adaptive_rag(
    question,
    candidate_k=8,
    final_k=4,
    max_corrections=1
)

print("QUESTION:")
print(result["question"])

print("\nSTRATEGY:")
print(result["strategy"])

print("\nANSWER:")
print(result["answer"])

print("\nRETRIEVAL GRADE:")
print(result["retrieval_grade"])

print("\nRETRIEVAL SCORE:")
print(round(
    result["retrieval_score"],
    3
))

print("\nANSWER SCORE:")
print(round(
    result["answer_score"],
    3
))

print("\nSOURCES:")

for source in result["sources"]:
    print("-", source["title"])

QUESTION:
What does ERR-401 mean?

STRATEGY:
SIMPLE

ANSWER:
ERR-401 indicates an authentication failure.

RETRIEVAL GRADE:
GOOD

RETRIEVAL SCORE:
1.0

ANSWER SCORE:
1.0

SOURCES:
- Authentication Errors
- BM25 Keyword Search
- Reranking
- Query Expansion


In [105]:
question = "What is that vector thing?"

result = adaptive_rag(
    question,
    candidate_k=8,
    final_k=4,
    max_corrections=1
)

print("QUESTION:")
print(result["question"])

print("\nSTRATEGY:")
print(result["strategy"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")

for source in result["sources"]:
    print("-", source["title"])

QUESTION:
What is that vector thing?

STRATEGY:
REWRITE

ANSWER:
Based on the provided evidence, a vector (or embedding) is a numerical representation that captures the semantic meaning of data or text. These numerical vectors are stored in vector databases and are used for semantic similarity search, allowing systems to retrieve information based on meaning rather than just exact keyword matching.

SOURCES:
- Vector Databases
- Embeddings
- Semantic Search
- Hybrid Search


In [106]:
question = """
How do embeddings work, how does BM25 work,
and why is reranking useful in RAG?
"""

result = adaptive_rag(
    question,
    candidate_k=8,
    final_k=4,
    max_corrections=1
)

print("QUESTION:")
print(result["question"])

print("\nSTRATEGY:")
print(result["strategy"])

print("\nANSWER:")
print(result["answer"])

print("\nRETRIEVAL GRADE:")
print(result["retrieval_grade"])

print("\nANSWER SCORE:")
print(round(
    result["answer_score"],
    3
))

QUESTION:

How do embeddings work, how does BM25 work,
and why is reranking useful in RAG?


STRATEGY:
DECOMPOSE

ANSWER:
Based on the provided evidence:

* **How embeddings work:** Embeddings are numerical vectors that represent the semantic meaning of text. Similar pieces of text have embeddings that are close to each other in vector space. 
* **How BM25 works:** BM25 is a keyword-based information retrieval algorithm that is useful for exact keywords, technical terminology, identifiers, and error codes.
* **Why reranking is useful in RAG:** Reranking acts as a second-stage retrieval process where a reranker gives candidate documents new relevance scores to improve the final ranking.

RETRIEVAL GRADE:
GOOD

ANSWER SCORE:
1.0


In [107]:
print("=" * 60)
print("ADAPTIVE RAG TRACE")
print("=" * 60)

for step in result["trace"]:

    print("\nSTEP:", step["step"])

    if "strategy" in step:
        print(
            "Strategy:",
            step["strategy"]
        )

    if "reason" in step:
        print(
            "Reason:",
            step["reason"]
        )

    if "query" in step:
        print(
            "Query:",
            step["query"]
        )

    if "queries" in step:
        print(
            "Queries:",
            step["queries"]
        )

    if "questions" in step:
        print(
            "Sub-questions:",
            step["questions"]
        )

    if "result" in step:
        print(
            "Result:",
            step["result"]
        )

ADAPTIVE RAG TRACE

STEP: router
Strategy: DECOMPOSE
Reason: The question contains three distinct, complex parts: embeddings, BM25, and reranking in RAG.

STEP: decomposition
Sub-questions: ['How do embeddings work?', 'How does BM25 work?', 'Why is reranking useful in RAG?']

STEP: retrieval

STEP: reranking

STEP: retrieval_evaluation
Result: {'grade': 'GOOD', 'score': 1.0, 'reason': 'The retrieved sources directly and clearly explain embeddings, BM25, and the purpose of reranking in RAG, covering all parts of the question.'}

STEP: generation

STEP: answer_evaluation
Result: {'supported': True, 'score': 1.0, 'reason': 'All parts of the answer (how embeddings work, how BM25 works, and why reranking is useful) are directly and fully supported by the provided sources.'}
